In [1]:
# Import necessary libraries
import yfinance as yf
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
%matplotlib inline


2025-01-22 20:10:31.328223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Extend the date range for Google stock data until 2028
end = datetime(2028, 12, 31)  # Extend end date to 2028
start = datetime(2004, 1, 1)  # Keep the original start date


In [3]:
# Download the data for the extended period (2004 to 2028)
stock = "GOOG"
google_data = yf.download(stock, start, end)


[*********************100%***********************]  1 of 1 completed


In [4]:
# Continue with the same preprocessing, scaling, etc.
Adj_close_price = google_data[['Close']]


In [5]:
# Apply scaling to the extended data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(Adj_close_price)


In [6]:
# Prepare x_data and y_data for training
x_data = []
y_data = []

for i in range(100, len(scaled_data)):
    x_data.append(scaled_data[i-100:i])
    y_data.append(scaled_data[i])


In [7]:
# Convert to numpy arrays
x_data, y_data = np.array(x_data), np.array(y_data)

In [8]:
# Split the data for training and testing
splitting_len = int(len(x_data) * 0.7)
x_train = x_data[:splitting_len]
y_train = y_data[:splitting_len]
x_test = x_data[splitting_len:]
y_test = y_data[splitting_len:]

In [9]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(128, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(LSTM(64, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

/Users/ansh/Music/AP/AP_Code/Major-Project/PY/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

In [11]:
# Train the model
model.fit(x_train, y_train, batch_size=1, epochs=2)

Epoch 1/2
3528/3528 ━━━━━━━━━━━━━━━━━━━━ 162s 45ms/step - loss: 2.1057e-04
Epoch 2/2
3528/3528 ━━━━━━━━━━━━━━━━━━━━ 199s 56ms/step - loss: 6.5487e-05


In [12]:
# Display model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 128)       │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │         1,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 352,859 (1.35 MB)

 Trainable params: 117,619 (459.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 235,240 (918.91 KB)

In [13]:
# Make predictions on the test data
predictions = model.predict(x_test)

48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step


In [14]:
# Inverse transform the predictions and test data
inv_predictions = scaler.inverse_transform(predictions)
inv_y_test = scaler.inverse_transform(y_test)

In [15]:
# Calculate the RMSE (Root Mean Squared Error)
rmse = np.sqrt(np.mean((inv_predictions - inv_y_test) ** 2))
rmse

5.241142005902955

In [16]:
# Prepare data for plotting
ploting_data = pd.DataFrame(
    {
        'original_test_data': inv_y_test.reshape(-1),
        'predictions': inv_predictions.reshape(-1)
    },
    index=google_data.index[splitting_len + 100:]
)

In [17]:
# Plot the original and predicted data
plot_graph((15, 6), ploting_data, 'test data')

NameError: name 'plot_graph' is not defined

In [ ]:
# Plot the whole data including predictions
plot_graph((15, 6), pd.concat([Adj_close_price[:splitting_len + 100], ploting_data], axis=0), 'whole data')

In [ ]:
# Save the model
model.save("Latest_stock_price_model.keras")